# 05 - MONAI 2D UNETR baseline

This notebook trains a single-frame MONAI UNETR on the existing EchoNet processed image-mask pairs. It preserves the baseline transforms and official EchoNet TRAIN/VAL/TEST partitioning.

In [ ]:
# Kaggle setup. Skip this cell if the session already satisfies requirements.txt.
%pip install -q monai opencv-python-headless pandas matplotlib scikit-learn tqdm pillow

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
import torch
from monai.data import DataLoader, Dataset


def find_project_root() -> Path:
    configured = os.environ.get('ECHONET_PROJECT_ROOT')
    candidates = [Path(configured)] if configured else []
    candidates.extend([Path.cwd(), Path.cwd().parent])
    if Path('/kaggle/input').exists():
        candidates.extend(path.parent.parent for path in Path('/kaggle/input').rglob('src/unetr_model.py'))
    for candidate in candidates:
        if (candidate / 'src' / 'unetr_model.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root containing src/unetr_model.py.')


def find_processed_dir(project_root: Path) -> Path:
    configured = os.environ.get('ECHONET_PROCESSED_DIR')
    candidates = [Path(configured)] if configured else []
    candidates.append(project_root / 'data' / 'processed')
    if Path('/kaggle/input').exists():
        candidates.extend(path.parent for path in Path('/kaggle/input').rglob('metadata.csv'))
    for candidate in candidates:
        if (candidate / 'images').is_dir() and (candidate / 'masks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate processed images/ and masks/.')


def find_raw_dir(project_root: Path) -> Path:
    configured = os.environ.get('ECHONET_RAW_DIR')
    candidates = [Path(configured)] if configured else []
    candidates.append(project_root / 'data' / 'raw' / 'EchoNet-Dynamic')
    if Path('/kaggle/input').exists():
        candidates.extend(path.parent for path in Path('/kaggle/input').rglob('FileList.csv'))
    for candidate in candidates:
        if (candidate / 'FileList.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate EchoNet FileList.csv for official splits.')


PROJECT_ROOT = find_project_root()
PROCESSED_DIR = find_processed_dir(PROJECT_ROOT)
RAW_DIR = find_raw_dir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import get_monai_transforms, load_processed_samples, split_by_echonet_filelist
from src.unetr_model import build_unetr
from src.unetr_train import (
    evaluate_unetr,
    fit_unetr,
    get_unetr_loss,
    plot_history,
    save_json,
    save_predictions,
    write_experiment_log,
)
from src.utils import set_seed

RUN_DIR = Path('/kaggle/working/outputs/runs/unetr') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'outputs' / 'runs' / 'unetr'
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
FIGURES_DIR = RUN_DIR / 'figures'
PREDICTIONS_DIR = FIGURES_DIR / 'predictions'
for directory in [RUN_DIR, CHECKPOINT_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Processed dataset: {PROCESSED_DIR}')
print(f'EchoNet metadata: {RAW_DIR}')
print(f'Device: {device}')
print(f'Output directory: {RUN_DIR}')

## Configuration

Run `smoke` first for one epoch on a small official-split subset. Switch to `full` after the complete workflow succeeds.

In [ ]:
RUN_MODE = 'smoke'  # change to 'full' for the complete experiment

SMOKE_CONFIG = {
    'run_mode': 'smoke',
    'seed': 42,
    'image_size': [112, 112],
    'epochs': 1,
    'batch_size': 4,
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'max_train_samples': 32,
    'max_val_samples': 16,
    'max_test_samples': 16,
    'feature_size': 16,
    'hidden_size': 384,
    'mlp_dim': 1536,
    'num_heads': 6,
    'dropout_rate': 0.0,
}

FULL_CONFIG = {
    'run_mode': 'full',
    'seed': 42,
    'image_size': [112, 112],
    'epochs': 50,
    'batch_size': 8,
    'learning_rate': 1e-4,
    'weight_decay': 1e-5,
    'num_workers': 2,
    'max_train_samples': None,
    'max_val_samples': None,
    'max_test_samples': None,
    'feature_size': 16,
    'hidden_size': 384,
    'mlp_dim': 1536,
    'num_heads': 6,
    'dropout_rate': 0.0,
}

config = SMOKE_CONFIG if RUN_MODE == 'smoke' else FULL_CONFIG
save_json(config, RUN_DIR / 'config.json')
config

## Load processed pairs and official EchoNet splits

The exact baseline loader and `split_by_echonet_filelist()` helper are reused. No random split fallback is introduced.

In [ ]:
samples = load_processed_samples(PROCESSED_DIR)
file_list = pd.read_csv(RAW_DIR / 'FileList.csv')
train_samples, val_samples, test_samples = split_by_echonet_filelist(samples, file_list)

matched_count = len(train_samples) + len(val_samples) + len(test_samples)
assert samples, 'No processed image-mask pairs were found.'
assert matched_count == len(samples), (
    f'{len(samples) - matched_count} processed samples did not match the official split.'
)
assert min(len(train_samples), len(val_samples), len(test_samples)) > 0, 'Every official split must contain samples.'

if config['max_train_samples'] is not None:
    train_samples = train_samples[:config['max_train_samples']]
if config['max_val_samples'] is not None:
    val_samples = val_samples[:config['max_val_samples']]
if config['max_test_samples'] is not None:
    test_samples = test_samples[:config['max_test_samples']]

print(f'Total processed pairs: {len(samples):,}')
print(f'Train: {len(train_samples):,}')
print(f'Validation: {len(val_samples):,}')
print(f'Test: {len(test_samples):,}')

In [ ]:
image_size = tuple(config['image_size'])
train_dataset = Dataset(train_samples, transform=get_monai_transforms(image_size=image_size, augment=True))
val_dataset = Dataset(val_samples, transform=get_monai_transforms(image_size=image_size, augment=False))
test_dataset = Dataset(test_samples, transform=get_monai_transforms(image_size=image_size, augment=False))

loader_kwargs = {
    'batch_size': config['batch_size'],
    'num_workers': config['num_workers'],
    'pin_memory': torch.cuda.is_available(),
    'persistent_workers': config['num_workers'] > 0,
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

sample_batch = next(iter(train_loader))
print(f"Image shape: {tuple(sample_batch['image'].shape)}")
print(f"Mask shape: {tuple(sample_batch['mask'].shape)}")
assert tuple(sample_batch['image'].shape[1:]) == (1, 112, 112)
assert tuple(sample_batch['mask'].shape[1:]) == (1, 112, 112)

## Train UNETR

In [ ]:
model = build_unetr(
    in_channels=1,
    out_channels=1,
    image_size=image_size,
    feature_size=config['feature_size'],
    hidden_size=config['hidden_size'],
    mlp_dim=config['mlp_dim'],
    num_heads=config['num_heads'],
    dropout_rate=config['dropout_rate'],
).to(device)
with torch.no_grad():
    sample_logits = model(sample_batch['image'][:1].to(device))
assert tuple(sample_logits.shape[1:]) == (1, 112, 112)
print(f'UNETR output shape: {tuple(sample_logits.shape)}')

loss_fn = get_unetr_loss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay'],
)

history = fit_unetr(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    device=device,
    epochs=config['epochs'],
    output_dir=RUN_DIR,
)
plot_history(history, FIGURES_DIR / 'training_curves.png')
display(history.tail())

## Best-checkpoint validation and held-out test evaluation

In [ ]:
best_checkpoint = torch.load(CHECKPOINT_DIR / 'best_model.pt', map_location=device)
model.load_state_dict(best_checkpoint['model_state_dict'])

val_metrics = evaluate_unetr(model, val_loader, loss_fn, device)
test_metrics = evaluate_unetr(model, test_loader, loss_fn, device)
save_json(test_metrics, RUN_DIR / 'test_metrics.json')

prediction_count = save_predictions(
    model=model,
    loader=test_loader,
    device=device,
    output_dir=PREDICTIONS_DIR,
    max_examples=10,
)
write_experiment_log(
    RUN_DIR / 'experiment_log.md',
    config=config,
    val_metrics=val_metrics,
    test_metrics=test_metrics,
)

print('Best-checkpoint validation metrics:', val_metrics)
print('Held-out test metrics:', test_metrics)
print(f'Saved prediction examples: {prediction_count}')

## Verify persistent outputs

In [ ]:
required_outputs = [
    CHECKPOINT_DIR / 'best_model.pt',
    CHECKPOINT_DIR / 'final_model.pt',
    FIGURES_DIR / 'training_curves.png',
    RUN_DIR / 'history.csv',
    RUN_DIR / 'config.json',
    RUN_DIR / 'test_metrics.json',
    RUN_DIR / 'experiment_log.md',
]
missing_outputs = [path for path in required_outputs if not path.exists()]
assert not missing_outputs, f'Missing required outputs: {missing_outputs}'

prediction_files = list(PREDICTIONS_DIR.glob('prediction_*.png'))
assert len(prediction_files) >= min(10, len(test_dataset)), 'Prediction figures are missing.'
all_output_files = sorted(path for path in RUN_DIR.rglob('*') if path.is_file())

print(f'Prediction count: {len(prediction_files):,}')
print(f'Total output files: {len(all_output_files):,}')
print(f'UNETR outputs: {RUN_DIR.resolve()}')
for path in all_output_files:
    print(f'  - {path}')